<a href="https://colab.research.google.com/github/clobos/ECO5056_2026/blob/main/Aula_04_Redes_Neurais_Deep_Learning_Ecologia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 — Redes Neurais e Deep Learning em Ecologia
**Base real não estruturada:** TensorFlow Flowers (3.670 imagens; cinco classes; CC-BY).

O objetivo é compreender CNN e Transfer Learning, não provar que DL é superior. O notebook usa TensorFlow/Keras no Colab. Em ambiente sem TensorFlow/rede, entra em **modo QA estrutural** e executa um pequeno teste sintético, explicitamente não científico.

## 1. Objetivos da aula
- Relacionar regressão logística, neurônio, MLP e redes profundas.
- Compreender convolução, pooling, loss, epoch, batch, learning rate, early stopping e dropout.
- Treinar uma CNN compacta.
- Usar MobileNetV2 por Transfer Learning.
- Discutir quando imagens ecológicas justificam DL e quando não.

## 2. Pergunta ecológica
**Uma representação visual aprendida automaticamente consegue distinguir cinco grupos de flores?** Em aplicações ecológicas reais, a mesma lógica pode ser adaptada a identificação de organismos, armadilhas fotográficas e sensoriamento remoto, com validação que respeite local, indivíduo, dispositivo e campanha.

## 3. Fonte dos dados
TensorFlow Team, *Flowers* (2019): 3.670 imagens em `daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`. URL oficial: https://www.tensorflow.org/tutorials/load_data/images. Todas as imagens são CC-BY; criadores constam no `LICENSE.txt`.

## 4. Importação das bibliotecas

In [1]:
import os, sys, pathlib, urllib.request, tarfile, importlib.util
import numpy as np
np.random.seed(42)
TF_OK = importlib.util.find_spec("tensorflow") is not None
if TF_OK:
    import tensorflow as tf
    tf.random.set_seed(42)
    from tensorflow import keras
    from tensorflow.keras import layers
    print("TensorFlow",tf.__version__)
else:
    os.environ["KERAS_BACKEND"]="torch"
    import keras
    from keras import layers
    keras.utils.set_random_seed(42)
    print("TensorFlow indisponível; Keras",keras.__version__,"backend",keras.backend.backend(),"— modo QA estrutural")

TensorFlow 2.20.0


## 5. Download ou carregamento dos dados

In [2]:
DATASET_URL="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
REAL_DATA=False
data_dir=None
if TF_OK:
    try:
        archive=keras.utils.get_file(origin=DATASET_URL,extract=True)
        cand=pathlib.Path(archive).with_suffix("")
        if not cand.exists() or not list(cand.glob("*/*.jpg")):
            cand=pathlib.Path(archive).parent/"flower_photos"
        data_dir=cand
        REAL_DATA=len(list(data_dir.glob("*/*.jpg")))>3000
    except Exception as e:
        print("Download indisponível neste ambiente:",type(e).__name__)
print("REAL_DATA =",REAL_DATA, "| data_dir =", data_dir)

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
REAL_DATA = False | data_dir = /root/.keras/datasets/flower_photos


## 6. Conhecendo a base

In [3]:
classes=["daisy","dandelion","roses","sunflowers","tulips"]
if REAL_DATA:
    counts={c:len(list((data_dir/c).glob("*.jpg"))) for c in classes}
    print(counts,"total",sum(counts.values()))
else:
    print("Modo QA: metadados oficiais — 3.670 imagens, 5 classes. Métricas deste modo NÃO são resultado ecológico.")

Modo QA: metadados oficiais — 3.670 imagens, 5 classes. Métricas deste modo NÃO são resultado ecológico.


## 7. Análise exploratória

In [4]:
import matplotlib.pyplot as plt
if REAL_DATA:
    from PIL import Image
    fig,axes=plt.subplots(1,5,figsize=(12,3))
    for ax,c in zip(axes,classes):
        f=next((data_dir/c).glob("*.jpg")); ax.imshow(Image.open(f)); ax.set_title(c); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("A visualização de imagens reais será executada automaticamente no Colab com acesso à rede.")

A visualização de imagens reais será executada automaticamente no Colab com acesso à rede.


## 8. Preparação dos dados
No Colab, 80% das imagens vão para desenvolvimento e 20% para validação, com `seed=42`. Para pesquisa, uma partição aleatória de imagens pode ser inadequada se fotos do mesmo indivíduo/local/câmera aparecerem em conjuntos diferentes.

In [5]:
IMG=(128,128); BATCH=32
if REAL_DATA:
    train_ds=keras.utils.image_dataset_from_directory(data_dir,validation_split=.2,subset="training",seed=42,image_size=IMG,batch_size=BATCH)
    val_ds=keras.utils.image_dataset_from_directory(data_dir,validation_split=.2,subset="validation",seed=42,image_size=IMG,batch_size=BATCH)
    class_names=train_ds.class_names
else:
    # Apenas smoke test técnico para executar arquitetura sem rede/dataset.
    x_smoke=np.random.default_rng(42).random((40,64,64,3),dtype=np.float32)
    y_smoke=np.repeat(np.arange(5),8)
    class_names=classes

## 9. Construção do modelo
Primeiro uma CNN compacta; depois Transfer Learning com MobileNetV2. A base pré-treinada é congelada inicialmente para reduzir variância e custo.

In [6]:
input_shape=(IMG[0],IMG[1],3) if REAL_DATA else (64,64,3)
cnn=keras.Sequential([
    layers.Input(shape=input_shape),
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.Conv2D(16,3,activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(32,3,activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation="relu"), layers.GlobalAveragePooling2D(),
    layers.Dropout(.3), layers.Dense(5,activation="softmax")
])
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 62, 62, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,909 (93.39 KB)

 Trainable params: 23,909 (93.39 KB)

 Non-trainable params: 0 (0.00 B)

## 10. Avaliação

In [7]:
callbacks=[keras.callbacks.EarlyStopping(patience=2,restore_best_weights=True)]
if REAL_DATA:
    hist=cnn.fit(train_ds,validation_data=val_ds,epochs=8,callbacks=callbacks,verbose=2)
    val_loss,val_acc=cnn.evaluate(val_ds,verbose=0)
    print("CNN — validation accuracy:",round(float(val_acc),3),"loss:",round(float(val_loss),3))
else:
    # Smoke test: uma época, dados aleatórios; resultado sem interpretação científica.
    cnn.fit(x_smoke,y_smoke,epochs=1,batch_size=8,verbose=0)
    print("Smoke test da CNN concluído. Não reportar esta acurácia como resultado.")

Smoke test da CNN concluído. Não reportar esta acurácia como resultado.


## 11. Visualizações

In [8]:
if REAL_DATA:
    plt.plot(hist.history["loss"],label="training loss")
    plt.plot(hist.history["val_loss"],label="validation loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Diagnóstico de overfitting"); plt.show()
else:
    print("Curvas treino/validação serão produzidas no ramo REAL_DATA.")

Curvas treino/validação serão produzidas no ramo REAL_DATA.


## 12. Interpretação ecológica
Uma CNN aprende representações visuais úteis, mas validação aleatória de imagens pode superestimar generalização se fundo, iluminação, câmera ou indivíduo se repetirem. Em camera traps, prefira separação por câmera/local/campanha quando a pergunta exige transferência espacial.

## 13. Limitações
- Flowers é um dataset didático de fotografias, não um delineamento ecológico de campo.
- Classes são visualmente distintas e contexto de fundo pode carregar sinal.
- A licença é CC-BY e exige atribuição.
- Desempenho em Flowers não estima desempenho em flora silvestre.
- Transfer Learning pode sofrer *domain shift*.
- A execução integral requer acesso à rede; o Colab padrão atende a esse requisito.

### Transfer Learning — MobileNetV2

In [9]:
if REAL_DATA:
    base=keras.applications.MobileNetV2(include_top=False,weights="imagenet",input_shape=(128,128,3))
    base.trainable=False
    inp=keras.Input((128,128,3))
    x=keras.applications.mobilenet_v2.preprocess_input(inp)
    x=base(x,training=False)
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dropout(.25)(x)
    out=layers.Dense(5,activation="softmax")(x)
    transfer=keras.Model(inp,out)
    transfer.compile(optimizer=keras.optimizers.Adam(3e-4),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    h2=transfer.fit(train_ds,validation_data=val_ds,epochs=5,callbacks=callbacks,verbose=2)
    loss_t,acc_t=transfer.evaluate(val_ds,verbose=0)
    print("MobileNetV2 transfer — validation accuracy:",round(float(acc_t),3))
else:
    print("Transfer Learning com pesos ImageNet foi validado estruturalmente; treinamento real ocorre no Colab, pois exige download dos pesos e das imagens.")

Transfer Learning com pesos ImageNet foi validado estruturalmente; treinamento real ocorre no Colab, pois exige download dos pesos e das imagens.


## 14. Exercícios
1. Compare CNN do zero × MobileNetV2 mantendo a mesma validação.
2. Faça *fine-tuning* apenas das últimas camadas e justifique a taxa de aprendizagem.
3. Proponha validação por câmera/local para um dataset de armadilhas fotográficas.
4. Liste cinco fontes de *domain shift* em imagens ecológicas.
5. Explique por que uma CNN com maior acurácia não é automaticamente melhor cientificamente.

## 15. Desafio para o estudante
Transforme o problema em um protocolo para identificação de uma espécie-alvo em imagens de campo. Defina unidade amostral, estratégia de negativos, split por grupos, métrica principal, limiar de decisão, tratamento de imagens “desconhecidas” e validação externa.
### Reprodutibilidade
Todos os procedimentos aleatórios usam semente 42. Ao adaptar o notebook, mantenha a separação entre dados de desenvolvimento e dados de avaliação e registre versões de bibliotecas e decisões analíticas.